In [46]:
import cv2
import os
from pathlib import Path
from tqdm import tqdm
def is_image_blurry(image_path, threshold=100.0):
    # 1. Load the image
    image = cv2.imread(image_path)
    
    # 2. Convert the image to grayscale (Laplacian works on a single channel)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # 3. Compute the Laplacian and calculate its variance
    focus_measure = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    # 4. Determine if it falls below the threshold
    is_blurry = focus_measure < threshold
    
    return is_blurry, focus_measure

blurred_images=[]

image_dir = Path("data/combined/crops/nut")
if not image_dir.is_dir():
    raise FileNotFoundError(f"Missing folder: {image_dir.resolve()}")
blurred_images = []
for name in tqdm(os.listdir(image_dir)):
    if not name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    full_path = image_dir / name  # important: full path, not just the filename
    blurry, score = is_image_blurry(str(full_path), threshold=100.0)
    if blurry:
        os.remove(full_path)
        blurred_images.append([name, score])

print(f"Number of Blurred Images: {len(blurred_images)}")
print(blurred_images)


100%|██████████| 2977/2977 [00:01<00:00, 2595.53it/s]

Number of Blurred Images: 131
[['CROP-005699.jpg', np.float64(67.27182889616752)], ['CROP-009884.jpg', np.float64(48.66987668213307)], ['CROP-008950.jpg', np.float64(89.72649442606726)], ['CROP-005719.jpg', np.float64(89.93418679502786)], ['CROP-010113.jpg', np.float64(59.32009323412491)], ['CROP-010107.jpg', np.float64(64.60754754944773)], ['CROP-010053.jpg', np.float64(37.60356152420181)], ['CROP-005260.jpg', np.float64(57.546032769348066)], ['CROP-009273.jpg', np.float64(85.77896177304248)], ['CROP-010153.jpg', np.float64(88.36540906014318)], ['CROP-005471.jpg', np.float64(16.43327998027774)], ['CROP-010066.jpg', np.float64(47.23542043887878)], ['CROP-005215.jpg', np.float64(95.21752297123419)], ['CROP-009961.jpg', np.float64(83.0172090405467)], ['CROP-009283.jpg', np.float64(70.47412240343624)], ['CROP-000859.jpg', np.float64(96.68781823857813)], ['CROP-008875.jpg', np.float64(51.43534766102883)], ['CROP-005485.jpg', np.float64(12.520298750192095)], ['CROP-005219.jpg', np.float64(5

In [23]:
image_dir = Path("data/train/images")
label_dir = Path("data/train/labels")

print(len(os.listdir(image_dir)))
print(len(os.listdir(label_dir)))

1771
1771


In [26]:
image_dir = Path("data/test/images")
label_dir = Path("data/test/labels")

print(len(os.listdir(image_dir)))
print(len(os.listdir(label_dir)))

181
181


In [29]:
image_dir = Path("data/valid/images")
label_dir = Path("data/valid/labels")

print(len(os.listdir(image_dir)))
print(len(os.listdir(label_dir)))

179
179


In [31]:
import os
import shutil
from pathlib import Path

IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png")

split_dirs = [
    Path("data/train"),
    Path("data/valid"),
    Path("data/test"),
]

combined_images = Path("data/combined/images")
combined_labels = Path("data/combined/labels")

# skip: leave existing files (safe to re-run the cell)
# overwrite: replace files in combined/
# error: stop on any name collision
ON_DUPLICATE = "skip"

# Set True to delete data/combined and rebuild from scratch
REBUILD_COMBINED = False

if REBUILD_COMBINED and Path("data/combined").exists():
    shutil.rmtree(Path("data/combined"))

combined_images.mkdir(parents=True, exist_ok=True)
combined_labels.mkdir(parents=True, exist_ok=True)

copied_images = 0
copied_labels = 0
skipped_images = 0

for split in split_dirs:
    image_dir = split / "images"
    label_dir = split / "labels"
    if not image_dir.is_dir():
        raise FileNotFoundError(f"Missing folder: {image_dir.resolve()}")

    for name in os.listdir(image_dir):
        if not name.lower().endswith(IMAGE_SUFFIXES):
            continue
        src_image = image_dir / name
        if not src_image.is_file():
            continue

        dst_image = combined_images / name
        label_name = f"{src_image.stem}.txt"
        dst_label = combined_labels / label_name

        if dst_image.exists():
            if ON_DUPLICATE == "error":
                raise FileExistsError(
                    f"Duplicate image name: {name} (from {split})"
                )
            if ON_DUPLICATE == "skip":
                skipped_images += 1
                continue
            # overwrite

        shutil.copy2(src_image, dst_image)
        copied_images += 1

        src_label = label_dir / label_name
        if src_label.is_file():
            if not dst_label.exists() or ON_DUPLICATE == "overwrite":
                shutil.copy2(src_label, dst_label)
                copied_labels += 1

print(f"Copied {copied_images} new images -> {combined_images.resolve()}")
print(f"Copied {copied_labels} new labels -> {combined_labels.resolve()}")
print(f"Skipped {skipped_images} images already in combined/")
print(f"Total in combined: {len(list(combined_images.iterdir()))} images")



Copied 2131 new images -> /home/abi/vison-model/Vision-Based Product Attribution and BOM Mapping/data/combined/images
Copied 2131 new labels -> /home/abi/vison-model/Vision-Based Product Attribution and BOM Mapping/data/combined/labels
Skipped 0 images already in combined/
Total in combined: 2131 images


In [43]:
import cv2
import os
from pathlib import Path
from tqdm import tqdm
def is_image_blurry(image_path, threshold=100.0):
    # 1. Load the image
    image = cv2.imread(image_path)
    
    # 2. Convert the image to grayscale (Laplacian works on a single channel)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # 3. Compute the Laplacian and calculate its variance
    focus_measure = cv2.Laplacian(gray, cv2.CV_64F).var()
    
    # 4. Determine if it falls below the threshold
    is_blurry = focus_measure < threshold
    
    return is_blurry, focus_measure

blurred_images=[]

image_dir = Path("data/combined/crops/gear")

if not image_dir.is_dir():
    raise FileNotFoundError(f"Missing folder: {image_dir.resolve()}")
blurred_images = []
for name in tqdm(os.listdir(image_dir)):
    if not name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue
    full_path = image_dir / name  # important: full path, not just the filename
    blurry, score = is_image_blurry(str(full_path), threshold=100.0)
    if blurry:
    
        blurred_images.append([name, score])

print(f"Number of Blurred Images: {len(blurred_images)}")
print(blurred_images)


100%|██████████| 2834/2834 [00:02<00:00, 1215.35it/s]

Number of Blurred Images: 116
[['CROP-003651.jpg', np.float64(76.8513832952231)], ['CROP-003650.jpg', np.float64(58.452384050213226)], ['CROP-007641.jpg', np.float64(79.97411734480453)], ['CROP-005794.jpg', np.float64(53.87654464591903)], ['CROP-005925.jpg', np.float64(28.611905091722168)], ['CROP-010400.jpg', np.float64(13.676879527855647)], ['CROP-006549.jpg', np.float64(70.98327682953156)], ['CROP-010660.jpg', np.float64(23.312332529035196)], ['CROP-006383.jpg', np.float64(50.10298189146519)], ['CROP-008196.jpg', np.float64(91.16409289360132)], ['CROP-008147.jpg', np.float64(67.80648666535794)], ['CROP-007672.jpg', np.float64(90.5576892526579)], ['CROP-000776.jpg', np.float64(86.75307276098584)], ['CROP-008117.jpg', np.float64(36.81847997157688)], ['CROP-001715.jpg', np.float64(54.36814463465037)], ['CROP-006380.jpg', np.float64(28.662808548181548)], ['CROP-000367.jpg', np.float64(78.57238305364973)], ['CROP-006376.jpg', np.float64(48.90402673807989)], ['CROP-006008.jpg', np.float64